# Admin Canvas - Creacion de assignments 2026-1

Notebook administrativo para crear assignments en Canvas usando la API.

**Usar con cuidado:** estas celdas pueden crear assignments reales en Canvas. Ejecuta primero el `dry_run` y revisa la tabla antes de activar la creacion real.

Flujo recomendado:
1. Cargar configuracion y credenciales.
2. Revisar que existan los grupos `Exámenes` y `Evaluaciones En Aula` en los cursos.
3. Ejecutar `dry_run=True`.
4. Si el plan es correcto, descomentar la celda de creacion real.
5. En Gradescope, vincular cada evaluacion al assignment de Canvas y hacer `Post Grades to Canvas`.


## 1. Imports y configuracion

In [7]:
import sys
import os
import yaml
import importlib
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Detectar raiz del repo tanto si el notebook se abre desde /notebooks como desde la raiz.
CWD = Path.cwd().resolve()
RAIZ = CWD if (CWD / "src").exists() else CWD.parent

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

load_dotenv(RAIZ / ".env")

import src.canvas_admin as canvas_admin
importlib.reload(canvas_admin)

from src.canvas_admin import (
    CanvasAdminClient,
    EXAMEN_PARCIAL,
    GRUPO_EVALUACIONES_AULA,
    GRUPO_EXAMENES,
    SIMULACRO_EP,
    evaluacion_en_aula,
    preparar_evaluaciones_aula,
    preparar_examen_parcial_2026_1,
)

CICLO = "2026-1"
with open(RAIZ / "config" / f"{CICLO}.yaml", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

print(f"Raiz: {RAIZ}")
print(f"Ciclo: {CONFIG['ciclo']}")
print(f"Canvas token cargado: {'si' if os.getenv('CANVAS_TOKEN') else 'NO'}")
GRUPOS_OBJETIVO = [GRUPO_EXAMENES, GRUPO_EVALUACIONES_AULA]

print(f"Grupos objetivo: {GRUPOS_OBJETIVO}")

Raiz: C:\Users\jsilvac\calculo-vectorial-pipeline
Ciclo: 2026-1
Canvas token cargado: si
Grupos objetivo: ['Exámenes', 'Evaluaciones en Aula']


## 2. Assignments que se prepararan

In [9]:
ea_ejemplo = evaluacion_en_aula(2)

pd.DataFrame([
    {
        "grupo_cursos": "auditorio",
        "secciones": list(CONFIG["canvas_api"]["courses_auditorio"].keys()),
        "assignment": SIMULACRO_EP.name,
        "puntos": SIMULACRO_EP.points_possible,
        "grupo_tareas": SIMULACRO_EP.assignment_group_name,
        "tipo_entrega": ", ".join(SIMULACRO_EP.submission_types),
    },
    {
        "grupo_cursos": "aula",
        "secciones": list(CONFIG["canvas_api"]["courses_aula"].keys()),
        "assignment": EXAMEN_PARCIAL.name,
        "puntos": EXAMEN_PARCIAL.points_possible,
        "grupo_tareas": EXAMEN_PARCIAL.assignment_group_name,
        "tipo_entrega": ", ".join(EXAMEN_PARCIAL.submission_types),
    },
    {
        "grupo_cursos": "aula",
        "secciones": list(CONFIG["canvas_api"]["courses_aula"].keys()),
        "assignment": f"{ea_ejemplo.name} (ejemplo futuro)",
        "puntos": ea_ejemplo.points_possible,
        "grupo_tareas": ea_ejemplo.assignment_group_name,
        "tipo_entrega": ", ".join(ea_ejemplo.submission_types),
    },
])

,grupo_cursos,secciones,assignment,puntos,grupo_tareas,tipo_entrega
0,auditorio,"[1, 2]",Simulacro EP,20,Exámenes,none
1,aula,"[11, 12, 13, 14, 21, 22, 23, 24]",Examen Parcial,23,Exámenes,none
2,aula,"[11, 12, 13, 14, 21, 22, 23, 24]",Evaluación en Aula 2 (ejemplo futuro),20,Evaluaciones en Aula,none


## 3. Revisar grupos de tareas en Canvas

In [14]:
client = CanvasAdminClient()

filas = []
for tipo, courses in [
    ("auditorio", CONFIG["canvas_api"]["courses_auditorio"]),
    ("aula", CONFIG["canvas_api"]["courses_aula"]),
]:
    for seccion, course_id in courses.items():
        try:
            grupos = client.listar_assignment_groups(course_id)
            nombres = grupos["name"].tolist() if "name" in grupos.columns else []
            fila = {
                "tipo": tipo,
                "seccion": seccion,
                "course_id": course_id,
                "grupos": ", ".join(nombres),
                "error": None,
            }
            for grupo in GRUPOS_OBJETIVO:
                fila[f"tiene_{grupo}"] = grupo in nombres
            filas.append(fila)
        except Exception as e:
            fila = {
                "tipo": tipo,
                "seccion": seccion,
                "course_id": course_id,
                "grupos": "",
                "error": str(e),
            }
            for grupo in GRUPOS_OBJETIVO:
                fila[f"tiene_{grupo}"] = False
            filas.append(fila)

df_grupos = pd.DataFrame(filas)
display(df_grupos)

cols_grupo = [f"tiene_{grupo}" for grupo in GRUPOS_OBJETIVO]
if not df_grupos[cols_grupo].all().all():
    print("AVISO: Al menos un curso no tiene todos los grupos objetivo.")
    print("Puedes crearlo manualmente en Canvas o usar create_group_if_missing=True en la creacion real.")

,tipo,seccion,course_id,grupos,error,tiene_Exámenes,tiene_Evaluaciones en Aula
0,auditorio,1,20189,"Tareas, Actividades Previas, Exámenes",None,True,False
1,auditorio,2,20205,"Tareas, Actividades Previas, Exámenes",None,True,False
2,aula,11,20245,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
3,aula,12,20246,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
4,aula,13,20247,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
5,aula,14,20248,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
6,aula,21,20249,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
7,aula,22,20250,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
8,aula,23,20251,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True
9,aula,24,20252,"Tareas, Evaluaciones en Aula, Exámenes",None,True,True


AVISO: Al menos un curso no tiene todos los grupos objetivo.
Puedes crearlo manualmente en Canvas o usar create_group_if_missing=True en la creacion real.


In [12]:
# Revisar qué cursos no tienen el grupo "Exámenes"

filas = []

for tipo, courses in [
    ("auditorio", CONFIG["canvas_api"]["courses_auditorio"]),
    ("aula", CONFIG["canvas_api"]["courses_aula"]),
]:
    for seccion, course_id in courses.items():
        grupos = client.listar_assignment_groups(course_id)
        nombres = grupos["name"].tolist() if "name" in grupos.columns else []

        filas.append({
            "tipo": tipo,
            "seccion": seccion,
            "course_id": course_id,
            "tiene_Exámenes": GRUPO_EXAMENES in nombres,
            "grupos": ", ".join(nombres),
        })

df_examenes_grupo = pd.DataFrame(filas)
display(df_examenes_grupo)

df_faltan_examenes = df_examenes_grupo[~df_examenes_grupo["tiene_Exámenes"]].copy()
display(df_faltan_examenes)

,tipo,seccion,course_id,tiene_Exámenes,grupos
0,auditorio,1,20189,True,"Tareas, Actividades Previas, Exámenes"
1,auditorio,2,20205,False,"Tareas, Actividades Previas"
2,aula,11,20245,False,"Tareas, Evaluaciones en Aula"
3,aula,12,20246,True,"Tareas, Evaluaciones en Aula, Exámenes"
4,aula,13,20247,False,"Tareas, Evaluaciones en Aula"
5,aula,14,20248,False,"Tareas, Evaluaciones en Aula"
6,aula,21,20249,False,"Tareas, Evaluaciones en Aula"
7,aula,22,20250,False,"Tareas, Evaluaciones en Aula"
8,aula,23,20251,False,"Tareas, Evaluaciones en Aula"
9,aula,24,20252,False,"Tareas, Evaluaciones en Aula"


,tipo,seccion,course_id,tiene_Exámenes,grupos
1,auditorio,2,20205,False,"Tareas, Actividades Previas"
2,aula,11,20245,False,"Tareas, Evaluaciones en Aula"
4,aula,13,20247,False,"Tareas, Evaluaciones en Aula"
5,aula,14,20248,False,"Tareas, Evaluaciones en Aula"
6,aula,21,20249,False,"Tareas, Evaluaciones en Aula"
7,aula,22,20250,False,"Tareas, Evaluaciones en Aula"
8,aula,23,20251,False,"Tareas, Evaluaciones en Aula"
9,aula,24,20252,False,"Tareas, Evaluaciones en Aula"


In [13]:
# CREACIÓN REAL del grupo "Exámenes" en cursos donde falta.
# Cambia CONFIRMAR_CREACION a "SI_CREAR_EXAMENES" solo cuando hayas revisado df_faltan_examenes.

CONFIRMAR_CREACION = "SI_CREAR_EXAMENES"

if CONFIRMAR_CREACION != "SI_CREAR_EXAMENES":
    print("No se creó nada. Revisa df_faltan_examenes y cambia CONFIRMAR_CREACION si estás seguro.")
else:
    filas_creadas = []

    for _, row in df_faltan_examenes.iterrows():
        creado = client.crear_assignment_group(
            course_id=row["course_id"],
            group_name=GRUPO_EXAMENES,
        )

        filas_creadas.append({
            "tipo": row["tipo"],
            "seccion": row["seccion"],
            "course_id": row["course_id"],
            "grupo_creado": creado.get("name"),
            "group_id": creado.get("id"),
        })

    df_grupos_creados = pd.DataFrame(filas_creadas)
    display(df_grupos_creados)

,tipo,seccion,course_id,grupo_creado,group_id
0,auditorio,2,20205,Exámenes,55206
1,aula,11,20245,Exámenes,55207
2,aula,13,20247,Exámenes,55208
3,aula,14,20248,Exámenes,55209
4,aula,21,20249,Exámenes,55210
5,aula,22,20250,Exámenes,55211
6,aula,23,20251,Exámenes,55212
7,aula,24,20252,Exámenes,55213


## 4. Dry run: revisar sin crear nada

In [15]:
df_plan = preparar_examen_parcial_2026_1(
    CONFIG,
    dry_run=True,
    create_group_if_missing=False,
)

display(df_plan)

,seccion,course_id,assignment,assignment_group,status,assignment_id,points_possible,published,payload
0,1,20189,Simulacro EP,Exámenes,dry_run_create,None,20,True,"{'name': 'Simulacro EP', 'points_possible': 20..."
1,2,20205,Simulacro EP,Exámenes,dry_run_create,None,20,True,"{'name': 'Simulacro EP', 'points_possible': 20..."
2,11,20245,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
3,12,20246,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
4,13,20247,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
5,14,20248,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
6,21,20249,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
7,22,20250,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
8,23,20251,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."
9,24,20252,Examen Parcial,Exámenes,dry_run_create,None,23,True,"{'name': 'Examen Parcial', 'points_possible': ..."


### Dry run opcional para futuras Evaluaciones en Aula

Cuando quieras preparar EA2, EA3, etc., usa esta celda cambiando la lista `numeros`.

In [ ]:
# df_plan_eas = preparar_evaluaciones_aula(
#     CONFIG,
#     numeros=[2, 3, 4, 5, 6],
#     dry_run=True,
#     create_group_if_missing=False,
# )
#
# display(df_plan_eas)

## 5. Creacion real

Descomenta esta celda solo despues de revisar el `dry_run`. No se crean duplicados: si el assignment ya existe por nombre exacto, el estado sera `exists`.

In [16]:
df_creacion = preparar_examen_parcial_2026_1(
     CONFIG,
     dry_run=False,
     create_group_if_missing=False,
 )

display(df_creacion)

,seccion,course_id,assignment,assignment_group,status,assignment_id,points_possible,published
0,1,20189,Simulacro EP,Exámenes,created,324885,20.0,True
1,2,20205,Simulacro EP,Exámenes,created,324886,20.0,True
2,11,20245,Examen Parcial,Exámenes,created,324887,23.0,True
3,12,20246,Examen Parcial,Exámenes,created,324888,23.0,True
4,13,20247,Examen Parcial,Exámenes,created,324889,23.0,True
5,14,20248,Examen Parcial,Exámenes,created,324890,23.0,True
6,21,20249,Examen Parcial,Exámenes,created,324891,23.0,True
7,22,20250,Examen Parcial,Exámenes,created,324892,23.0,True
8,23,20251,Examen Parcial,Exámenes,created,324893,23.0,True
9,24,20252,Examen Parcial,Exámenes,created,324894,23.0,True


## 6. Despues de crear

1. En Gradescope, vincula `Simulacro EP` a Canvas en Teoria 1 y Teoria 2.
2. En Gradescope, vincula `Examen Parcial` a Canvas en Teorias 11, 12, 13, 14, 21, 22, 23 y 24.
3. Ejecuta `Post Grades to Canvas`.
4. Agrega esos nombres a `config/2026-1.yaml` en `assignments_auditorio` y `assignments_aula`.
5. Corre el notebook principal `2026-1_exploracion.ipynb` para descargar `SExP` y `ExP`.